# 3D Data Deep Learning With PyTorch

Notebook implements the project from Part to of "Deep Learning with PyTorch". From; https://isip.piconepress.com/courses/temple/ece_4822/resources/books/Deep-Learning-with-PyTorch.pdf. This Notebook try to convert the written code in Patterns which can be reused for further development.

## Preparing a project

## Data Loading

### Training and validation sets

Task to solve in this project setup setp:
- Loading and processing raw data files
- Implementing a Python class to represent our data
- Converting our data into a format usable by PyTorch
- Visualizing the training and validation data

### Unifying our annotation and candidate data

### Data Pre-processing

The first target is to split the data in a training and validation set. LUNA16 provides us with annotations and candidate data.

In [4]:
import csv
annotations = []
candidates = []
with open('/media/lennart/LaCie/datasets/LUNA16/candidates.csv') as f:
    candidates = f.readlines()

with open('/media/lennart/LaCie/datasets/LUNA16/annotations.csv') as f:
    annotations = f.readlines()

print(f"lenght candidtes: {len(candidates)}, length annotations: {len(annotations)}")

lenght candidtes: 551066, length annotations: 1187


Meaning 551,066 coordinates can be considered as being a nodule, while 1,187 are actually nodules.

#### Unifying annotation and candidate data
Stitching candidate and annotation data together.

In [6]:
DATASET_PATH = '/media/lennart/LaCie/datasets/LUNA16/'

In [7]:
from collections import namedtuple
CandidateInfoTuple = namedtuple(
'CandidateInfoTuple','isNodule_bool, diameter_mm, series_uid, center_xyz',
)

In [ ]:
@functools.lru_cache(1)
def getCandidateInfoList(requireOnDisk_bool=True):
    mhd_list = glob.glob(os.path.join(PATH_TO_DATASET, 'subset*/*.mhd'))
    presentOnDisk_set = {os.path.split(p)[-1][:4] for p in mhd_list} # all the data that is present on disk

    daimeter_dict = {}
    with open(os.path.join(PATH_TO_DATASET, 'annotations.csv'), "r") as f:
        for row in list(csv.reader(f))[1:]:
            series_uid = row[0]
            annotationCenter_xyz = tuple([float(x) for x in row[1:4]])
            annotationDiameter_mm = float(row[4])
            diameter_dict.setdefault(series_uid, []).append(
                (annotationCenter_xyz, annotationDiameter_mm)
            )

    with open(os.path.join(PATH_TO_DATASET, 'candidates.csv'), "r") as f:
        for row in list(csv.reader(f))[1:]:
            series_uid = row[0]

            if series_uid not in presentOnDisk_set and requireOnDisk_bool:
                continue

            isNodule_bool = bool(int(row[4]))
            candidateCenter_xyz = tuple([float(x) for x in row[1:4]])
            candidateDiameter_mm = 0.0
            
            for annotation_tup in diameter_dict.get(series_uid, []):
                annotationCenter_xyz, annotationDiameter_mm = annotation_tup
                
                for i in range(3):
                    delta_mm = abs(candidateCenter_xyz[i] - annotationCenter_xyz[i])
                    
                    if delta_mm > annotationDiameter_mm / 4: # candidate and center should not be too far apart
                        break
                    else:
                        candidateDiameter_mm = annotationDiameter_mm
                        break
                            
            candidateInfo_list.append(CandidateInfoTuple(
                isNodule_bool,
                candidateDiameter_mm,
                series_uid,
                candidateCenter_xyz,
            ))
            candidateInfo_list.sort(reverse=True) # samples starting with largest first
            return candidateInfo_list
                

#### Parsing the domain-specific data (CT-scans)

In [ ]:
import SimpleITK as sitk

class Ct:
    def __init__(self, series_uid):
        mhd_path = glob.glob(
            os.path.join(DATASET_PATH, 'subset*/{}.mhd'.format(series_uid))
        )[0]
        ct_mhd = sitk.ReadImage(mhd_path)
        ct_a = np.array(sitk.GetArrayFromImage(ct_mhd), dtype=np.float32)
        ct_a.clip(-1000, 1000, ct_a) # In LUNA air is -1,000 HU, water is 0 HU and bone is 1,000 HU
        self.series_uid = series_uid
        self.hu_a = ct_a
        
        self.origin_xyz = XyzTuple(*ct_mhd.GetOrigin())
        self.vxSize_xyz = XyzTuple(*ct_mhd.GetSpacing())
        self.direction_a = np.array(ct_mhd.GetDirection()).reshape(3, 3)

    def getRawCandidate(self, center_xyz, width_irc):
        center_irc = xyz2irc(
            center_xyz,
            self.origin_xyz,
            self.vxSize_xyz,
            self.direction_a,
        )
        slice_list = []
        for axis, center_val in enumerate(center_irc):
            start_ndx = int(round(center_val - width_irc[axis]/2))
            end_ndx = int(start_ndx + width_irc[axis])
            slice_list.append(slice(start_ndx, end_ndx))
        ct_chunk = self.hu_a[tuple(slice_list)]
        return ct_chunk, center_irc

#### Coordinate System Transformation

In [ ]:
IrcTuple = collections.namedtuple('IrcTuple', ['index', 'row', 'col'])
XyzTuple = collections.namedtuple('XyzTuple', ['x', 'y', 'z'])

def irc2xyz(coord_irc, origin_xyz, vxSize_xyz, direction_a):
    cri_a = np.array(coord_irc)[::-1] # Swapping the order
    origin_a = np.array(origin_xyz)
    vxSize_a = np.array(vxSize_xyz)
    coords_xyz = (direction_a @ (cri_a * vxSize_a)) + origin_a
    return XyzTuple(*coords_xyz)
    
def xyz2irc(coord_xyz, origin_xyz, vxSize_xyz, direction_a):
    origin_a = np.array(origin_xyz)
    vxSize_a = np.array(vxSize_xyz)
    coord_a = np.array(coord_xyz)
    cri_a = ((coord_a - origin_a) @ np.linalg.inv(direction_a)) / vxSize_a
    cri_a = np.round(cri_a)
    return IrcTuple(int(cri_a[2]), int(cri_a[1]), int(cri_a[0]))

#### PyTorch Dataset implementation

In [ ]:
class LunaDataset(Dataset):
    def __init__(self,
                 val_stride=0,
                 isValSet_bool=None,
                 series_uid=None,
                ):
        self.candidateInfo_list = copy.copy(getCandidateInfoList())
        if series_uid: # For debugging or visualization
            self.candidateInfo_list = [
                x for x in self.candidateInfo_list if x.series_uid == series_uid
            ]
        if isValSet_bool:
            assert val_stride > 0, val_stride
            self.candidateInfo_list = self.candidateInfo_list[::val_stride]
            assert self.candidateInfo_list
        elif val_stride > 0:
            del self.candidateInfo_list[::val_stride]
            assert self.candidateInfo_list
            
    def __len__(self):
        return len(self.candidateInfo_list)

    def __getitem__(self, ndx):
        candidateInfo_tup = self.candidateInfo_list[ndx]
        width_irc = (32, 48, 48)
        candidate_a, center_irc = getCtRawCandidate(
            candidateInfo_tup.series_uid,
            candidateInfo_tup.center_xyz,
            width_irc,
        )
        candidate_t = torch.from_numpy(candidate_a)
        candidate_t = candidate_t.to(torch.float32)
        candidate_t = candidate_t.unsqueeze(0)
        pos_t = torch.tensor([
            not candidateInfo_tup.isNodule_bool,
            candidateInfo_tup.isNodule_bool
        ], dtype=torch.long,)
        return (
           candidate_t, 1((CO10-1))
           pos_t, 1((CO10-2))
           candidateInfo_tup.series_uid,
           torch.tensor(center_irc),
    )
            
    

In [ ]:
@functools.lru_cache(1, typed=True)
def getCt(series_uid):
    return Ct(series_uid)
    
@raw_cache.memoize(typed=True)
def getCtRawCandidate(series_uid, center_xyz, width_irc):
    ct = getCt(series_uid)
    ct_chunk, center_irc = ct.getRawCandidate(center_xyz, width_irc)
    return ct_chunk, center_irc

## Segmentation

## Grouping

## Classification

Basic go to strategy:
1. Init model
2. Init data loaders
3. Loop over epochs
    1. Training Loop
        1. Load batch tuple
        2. Classify batch
        3. Calculate loss
        4. Record metrics
        5. Update weights
   2. Validation Loop
        1. Load batch tuple
        2. Classify batch
        3. Calculate loss
        4. Record Metrics

### Application entry
Should be a standalone script that runs from project root

In [ ]:
def run(app, *argv):
    argv = list(argv)
    argv.insert(0, '--num-workers=4')
    log.info("Running: {}({!r}).main()".format(app, argv))
    app_cls = importstr(*app.rsplit('.', 1))
    app_cls(argv).main()
    log.info("Finished: {}.{!r}).main()".format(app, argv))

In [ ]:
class LunaTrainingApp:
    def main(self):
        train_dl = self.initTrainDl()
        val_dl = self.initValDl()
        for epoch_ndx in range(1, self.cli_args.epochs + 1):
            trnMetrics_t = self.doTraining(epoch_ndx, train_dl)
            self.logMetrics(epoch_ndx, 'trn', trnMetrics_t)
            valMetrics_t = self.doValidation(epoch_ndx, val_dl)

    def computeBatchLoss(self, batch_ndx, batch_tup, batch_size, metrics_g):
        input_t, label_t, _series_list, _center_list = batch_tup
        input_g = input_t.to(self.device, non_blocking=True)
        label_g = label_t.to(self.device, non_blocking=True)
        logits_g, probability_g = self.model(input_g)
        loss_func = nn.CrossEntropyLoss(reduction='none')
        loss_g = loss_func(
            logits_g,
            label_g[:,1],
        )
        return loss_g.mean()

    def doTraining(self, epoch_ndx, train_dl):
        self.model.train()
        trnMetrics_g = torch.zeros(
            METRICS_SIZE,
            len(train_dl.dataset),
            device=self.device,
        )
    
        batch_iter = enumerateWithEstimate(
            train_dl,
            "E{} Training".format(epoch_ndx),
            start_ndx=train_dl.num_workers,
        )
    
        for batch_ndx, batch_tup in batch_iter:
            self.optimizer.zero_grad()
            loss_var = self.computeBatchLoss(
                batch_ndx,
                batch_tup,
                train_dl.batch_size,
                trnMetrics_g
            )
            loss_var.backward()
            self.optimizer.step()
        self.totalTrainingSamples_count += len(train_dl.dataset)
        return trnMetrics_g.to('cpu')

    def doValidation(self, epoch_ndx, val_dl):
        with torch.no_grad():
            self.model.eval()
            valMetrics_g = torch.zeros(
                METRICS_SIZE,
                len(val_dl.dataset),
                device=self.device,
            )
        
        batch_iter = enumerateWithEstimate(
            val_dl,
            "E{} Validation ".format(epoch_ndx),
            start_ndx=val_dl.num_workers,
        )
        for batch_ndx, batch_tup in batch_iter:
            self.computeBatchLoss(
                batch_ndx, batch_tup, val_dl.batch_size, valMetrics_g)
        return valMetrics_g.to('cpu')
    

if __name__ == '__main__':
    LunaTrainingApp().main()

### Training Class
Requirements for a training class:
1. Initalize the Optimizer
2. Initalize the Model
3. Initalize the Train Dataloader
4. Initalize the Validation Dataloader

In [ ]:
class LunaTrainingApp:
    def __init__(self, sys_argv=None):
        if sys_argv is None:
            sys_argv = sys.argv[1:]
            parser = argparse.ArgumentParser()
            parser.add_argument('--num-workers',
                                help='Number of worker processes for background data loading',
                                default=8,
                                type=int,
                               )
        self.use_cuda = torch.cuda.is_available()
        self.device = torch.device("cuda" if self.use_cuda else "cpu")
        self.model = self.initModel()
        self.optimizer = self.initOptimizer()
        
    def initModel(self):
        model = LunaModel()
        if self.use_cuda:
            log.info("Using CUDA; {} devices.".format(torch.cuda.device_count()))
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(self.device)
        return model

    def initOptimizer(self):
        return SGD(self.model.parameters(), lr=0.001, momentum=0.99)

    def initTrainDl(self):
        train_ds = LunaDataset(
            val_stride=10,
            isValSet_bool=False,
        )
        
        batch_size = self.cli_args.batch_size
        if self.use_cuda:
            batch_size *= torch.cuda.device_count()
        
        train_dl = DataLoader(
            train_ds,
            batch_size=batch_size,
            num_workers=self.cli_args.num_workers,
            pin_memory=self.use_cuda,
        )

    def initValDl(self):
        val_ds = LunaDataset(
            val_stride=10,
            isValSet_bool=True,
        )
        
        batch_size = self.cli_args.batch_size
        if self.use_cuda:
            batch_size *= torch.cuda.device_count()
        
        val_dl = DataLoader(
            val_ds,
            batch_size=batch_size,
            num_workers=self.cli_args.num_workers,
            pin_memory=self.use_cuda,
        )

### LUNA Model Class
Classification models often have a structure that consists of a *tail*, a *backbone* (or
body), and a *head*. The *tail* is the first few layers that process the input to the network.
These early layers often have a different structure or organization than the rest of the
network, as they must adapt the input to the form expected by the *backbone*. Here we
use a simple batch normalization layer, though often the tail contains convolutional
layers as well. The *head* of the network takes the output from the backbone and converts it
into the desired output form. 

In [ ]:
class LunaBlock(nn.Module):
    def __init__(self, in_channels=1, conv_channels=8):
        super().__init__()
        self.tail_batchnorm = nn.BatchNorm3d(1)
        self.block1 = LunaBlock(in_channels, conv_channels)
        self.block2 = LunaBlock(conv_channels, conv_channels * 2)
        self.block3 = LunaBlock(conv_channels * 2, conv_channels * 4)
        self.block4 = LunaBlock(conv_channels * 4, conv_channels * 8)
        self.head_linear = nn.Linear(1152, 2)
        self.head_softmax = nn.Softmax(dim=1)

        self.conv1 = nn.Conv3d(
            in_channels, 
            conv_channels, 
            kernel_size=3, 
            padding=1, 
            bias=True,)
        self.relu1 = nn.ReLU(inplace=True) 1((CO5-1))
        self.conv2 = nn.Conv3d(
            conv_channels, conv_channels, kernel_size=3, padding=1, bias=True,
        )
        self.relu2 = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(2, 2)

def forward(self, input_batch):
    bn_output = self.tail_batchnorm(input_batch)
    block_out = self.block1(bn_output)
    block_out = self.block2(block_out)
    block_out = self.block3(block_out)
    block_out = self.block4(block_out)
    conv_flat = block_out.view(
        block_out.size(0),
        -1,
    )
    linear_output = self.head_linear(conv_flat)
    return linear_output, self.head_softmax(linear_output)

def _init_weights(self):
    for m in self.modules():
        if type(m) in {
            nn.Linear,
            nn.Conv3d,
        }:
            nn.init.kaiming_normal_(
                m.weight.data, a=0, mode='fan_out', nonlinearity='relu',
            )
        if m.bias is not None:
            fan_in, fan_out = \
                nn.init._calculate_fan_in_and_fan_out(m.weight.data)
            bound = 1 / math.sqrt(fan_out)
            nn.init.normal_(m.bias, -bound, bound

### Estimate Time consumption

The function `enumerateWithEstimate` function enables to gather information how long a certain epoch should take.

In [ ]:
for i, _ in enumerateWithEstimate(list(range(234)), "sleeping"):
    time.sleep(random.random())

### Visualization

Using Tensorboard helps to visualize and debug training. Logging and Tensorflow is required to make this work.

In [1]:
tensorboard --logdir runs/

SyntaxError: invalid syntax (2872269074.py, line 1)

Tensorboard requires certain loggin format so that it can consume it properly. We will use SummaryWriter to build log files in the required format.

In [ ]:
def initTensorboardWriters(self):
    if self.trn_writer is None:
        log_dir = os.path.join('runs', self.cli_args.tb_prefix, self.time_str)
        
        self.trn_writer = SummaryWriter(
            log_dir=log_dir + '-trn_cls-' + self.cli_args.comment)
        self.val_writer = SummaryWriter(
            log_dir=log_dir + '-val_cls-' + self.cli_args.comment)

In [ ]:
for key, value in metrics_dict.items():
        writer.add_scalar(key, value, self.totalTrainingSamples_count)

### Debugging

To debug not expected outcomes it helps to gather metrics during training, therefore integrate for classification F1, Precision and Recall into the training process.

In [ ]:
log.info(
("E{} {:8} {loss/neg:.4f} loss, "
    + "{correct/neg:-5.1f}% correct ({neg_correct:} of {neg_count:})"
).format(
    epoch_ndx,
    mode_str + '_neg',
    neg_correct=neg_correct,
    neg_count=neg_count,
    **metrics_dict,
    )
)

log.info(
("E{} {:8} {loss/all:.4f} loss, "
    + "{correct/all:-5.1f}% correct, "
    + "{pr/precision:.4f} precision, "
    + "{pr/recall:.4f} recall, "
    + "{pr/f1_score:.4f} f1 score"
).format(
    epoch_ndx,
    mode_str,
    **metrics_dict,
    )
)

## Augmentation and Balancing the dataset

### Balance 
One problem a dataset can encounter is that there are too less examples to be true positive samples with result. The model should not be able to exploit the larger structure of the tests to answer things correctly.

#### Samplers
Sampling can involve deciding strategies, as of 1-to-1 sampling etc.

In [ ]:
class DatasetWithSampling:
    def __init__(self,
                 val_stride=0,
                 isValSet_bool=None,
                 ratio_int=0 # e.g. 2 meaning a 2:1 ratio of negative to positive
                ):
        self.ratio_int = ratio_int
        # ...
        self.negtive_list = [
            nt for nt in self.candidateInfo_list if not nt.isNodule_bool
        ]
        self.pos_list = [
            nt for nt in self.candidateInfo_list if if nt.isNodule_bool
        ]
        # ...

    def shuffleSamples(self):
        if self.ratio_int:
            random.shuffle(self.negative_list)
            random.shuffle(self.pos_list)

    def __getitem__(self, ndx):
        if self.ratio_int:
            pos_ndx = ndx // (self.ratio_int + 1)

            if ndx % (self.ratio_int + 1): # Nonzero remainder means should be negtive sample 
                neg_ndx = ndx -1 - pos_ndx
                neg_ndx %= len(self.negtative_list) # Overflow results in wraparound
                candidateInfo_tup = self.negative_list[neg_ndx]
            else:
                pos_ndx %= len(self.pos_list) # Overflow results in wraparound
                candidateInfo_tup = self.pos_list[pos_ndx]
        else:
            candidateInfo_tup = self.candidateInfo_list[ndx] # Returns the Nth sample if not balancing classes
        

### Data Augmentation
Data augmentation can be one solution to prevend overfitting. Data augementation should happen after caching not before.

In [ ]:
def getCtAugmentedCandidate(
    augmentation_dict,
    series_uid, center_xyz, width_irc,
    use_cache=True):
    if use_case:
        ct_chunk, center_irc = \ getRawCandidate(series_uid, center_xyz, width_ird)
    else:
        ct = getCt(series_uid)
        ct_chunk, center_irc = ct.getRawCandidate(center_xyz, width_irc)

    ct_t = torch.tensor(ct_chunk).unsqueeze(0).unsqueeze(0).to(torch.float32)
    transform_t = torch.eye(4)
    # ...
    affine_t = F.affine_grid(
        transform_t[:3].unsqueeze(0).to(torch.float32),
        ct_t.size(),
        align_corners=False,
    )
    augmented_chunk = F.grid_sample(
        ct_t,
        affine_t,
        padding_mode='border',
        align_corners=False,
    ).to('cpu')
    return augmented_chunk[0], center_irc
    

#### Mirroring
Mirroring a sample, we keep the pixel values exactly the same and only change
the orientation of the image. 

In [ ]:
for i in range(3):
    if 'flip' in augmentation_dict:
        if random.random() > 0.5:
            transform_t[i,i] *= -1

#### Shifting by random offset
Data will be resampled
using trilinear interpolation, which can introduce some slight blurring. Voxels at the
edge of the sample will be repeated, which can be seen as a smeared, streaky section
along the border.

In [ ]:
for i in range(3):
    if 'offset' in augmentation_dict:
        offset_float = augemetnation_dict['offset']
        random_float = (random.random() * 2 -1)
        transform_t[i,3] = offset_float * random_float

#### Scaling
Result in the same repeated edge voxels we just mentioned when discussing shifting
the sample.

In [ ]:
for i in range(3):
# ... line 175
    if 'scale' in augmentation_dict:
        scale_float = augmentation_dict['scale']
        random_float = (random.random() * 2 - 1)
        transform_t[i,i] *= 1.0 + scale_float * random_float

#### Rotating
Resample or data so that our resoltion along the index-axis is the same as along the other two.

In [ ]:
if 'rotate' in augmentation_dict:
    angle_rad = random.random() * math.pi * 2
    s = math.sin(angle_rad)
    c = math.cos(angle_rad)
    
    rotation_t = torch.tensor([
        [c, -s, 0, 0],
        [s, c, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ])
    transform_t @= rotation_t

#### Noise
Adding too much noise to the sample, will swamp the real data data and make it effectively impossible to classify.

In [ ]:
if 'noise' in augmentation_dict:
    noise_t = torch.randn_like(augmented_chunk)
    noise_t *= augmentation_dict['noise']
    
    augmented_chunk += noise_t